# Agent Types

Toda a ideia de utilizarmos agente é termos um sistema que saiba que possue ferramntas diferentes em mãos para utilizar e aumentar suas capacidades e as utilize nos momentos adequados. Para que isso seja possível, a OpenAI fez diversos processos de fine tunning em seus modelos, de forma que eles consiga gerar outputs paronizadas para a utilização das ferramentas. Entretanto, muito desse processo pode ser feito, ou melhorado, através de Engenharia de Prompts, e por isso, diversos tipos de agentes (Agent Types) foram criados na literatura e no framework LangChain. Cada um tenta otimizar a utilização das ferramentas e vamos falar sobre os dois mais utilizados: Toll Calling e ReAct. Para verificar todos, você pode acessar a página: https://python.langchain.com/v0.1/docs/modules/agents/agent_types/.

## Tool calling agent

Tool calling permite que um modelo detecte quando uma ou mais ferramentas devem ser acionadas e responda com os dados que devem ser passados para essas ferramentas. Em uma chamada de API, você pode descrever ferramentas e fazer com que o modelo escolha inteligentemente gerar um objeto estruturado, como JSON, contendo argumentos para acionar essas ferramentas. O objetivo das APIs de ferramentas é retornar chamadas de ferramentas válidas e úteis de forma mais confiável do que o que pode ser feito usando uma API genérica de conclusão de texto ou chat.

Podemos aproveitar essa saída estruturada, combinada com o fato de que você pode vincular várias ferramentas a um modelo de chat que chama ferramentas e permitir que o modelo escolha qual chamar, para criar um agente que repetidamente chama ferramentas e recebe resultados até que uma consulta seja resolvida.

Esta é uma versão mais generalizada do agente de ferramentas do OpenAI, que foi projetada para o estilo específico de chamada de ferramentas da OpenAI. Ela usa a interface ToolCall do LangChain para suportar uma gama mais ampla de implementações de provedores, como Anthropic, Google Gemini e Mistral, além do OpenAI.

In [3]:
from langchain_experimental.tools import PythonAstREPLTool

tools = [PythonAstREPLTool()]

In [7]:

from langchain_experimental.tools import PythonAstREPLTool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")
tools = [PythonAstREPLTool()]
# system_msg = '''Você é um assistente amigável.
# Certifique-se de usar a ferramenta PythonAstREPLTool para auxiliar a responder as perguntas'''
system_msg = """Você é um agente projetado para escrever e executar código Python para responder perguntas.
Você tem acesso a um REPL Python, que pode usar para executar código Python.
Se encontrar um erro, depure o código e tente novamente.
Use apenas a saída do seu código para responder à pergunta.
Você pode conhecer a resposta sem executar nenhum código, mas deve ainda assim executar o código para obter a resposta.
Se não parecer possível escrever código para responder à pergunta, simplesmente retorne "Não sei" como a resposta."""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_msg),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}')
])

# chain = prompt | model

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({'input': 'Quantas letras tem a palavra Genival?'})




> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "print(len('Genival'))"}`


7
A palavra Genival tem 7 letras.

> Finished chain.


{'input': 'Quantas letras tem a palavra Genival?',
 'output': 'A palavra Genival tem 7 letras.'}

## ReAct agent (Reason + Act)

É uma técnica de Engenharia de Promots criada para permitir que os LLMs interajam com ferramentas externas para recuperar informações adicionais que levam a respostas mais confiáveis e factuais.


In [12]:
from langchain.agents import create_react_agent

https://smith.langchain.com/hub/

In [11]:
from langchain import hub

prompt = hub.pull('hwchase17/react')
prompt

print(prompt.template)

c:\Users\geniv\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [14]:
from langchain_experimental.tools import PythonAstREPLTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub


tools = [PythonAstREPLTool()]
prompt = hub.pull('hwchase17/react')
prompt

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")
agent = create_react_agent(chat, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({'input': 'Qual é o décimo valor da sequência fibonacci?'})

c:\Users\geniv\AppData\Local\Programs\Python\Python313\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(




> Entering new AgentExecutor chain...
I need to find the 10th Fibonacci number. I can use Python to calculate this.
Action: python_repl_ast
Action Input:
```python
def fibonacci(n):
    if n <= 1:
        return n
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b

print(fibonacci(10))
```55
I have calculated the 10th Fibonacci number.
Final Answer: 55


> Finished chain.


{'input': 'Qual é o décimo valor da sequência fibonacci?', 'output': '55'}

In [15]:
from langchain_experimental.tools import PythonAstREPLTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain.prompts import PromptTemplate



tools = [PythonAstREPLTool()]
prompt = PromptTemplate.from_template(
'''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of {tool_names}
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''
)

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")
agent = create_react_agent(chat, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({'input': 'Qual é o décimo valor da sequência fibonacci?'})





> Entering new AgentExecutor chain...
I need to find the 10th Fibonacci number. I can use a Python script to calculate this.
Action: python_repl_ast
Action Input:
```python
def fibonacci(n):
  if n <= 1:
    return n
  else:
    a, b = 0, 1
    for _ in range(2, n + 1):
      a, b = b, a + b
    return b

print(fibonacci(10))
```55
I now know the final answer
Final Answer: 55


> Finished chain.


{'input': 'Qual é o décimo valor da sequência fibonacci?', 'output': '55'}